# Experiment 3 Pose Roads

Reproducibility notebook for the CNN experiments in *Are You Thinking What I'm Thinking?*.

- Expected image folders are documented in `data/README.md`.
- Set `CNN_DATA_ROOT`, `POSE_DATA_ROOT`, or `ROAD_DATA_ROOT` as environment variables if your images live elsewhere.
- The notebook uses pretrained ImageNet weights and extracts pre-final-layer activations.
- Outputs have been cleared from the repository version; run cells top-to-bottom to reproduce results.


In [ ]:
# Repository paths and reproducibility settings
from pathlib import Path
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "cnns").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")

REPO_ROOT = find_repo_root()
CNN_DATA_ROOT = Path(os.environ.get("CNN_DATA_ROOT", REPO_ROOT / "data" / "cnn_images"))
POSE_DATA_ROOT = Path(os.environ.get("POSE_DATA_ROOT", CNN_DATA_ROOT / "Pose"))
ROAD_DATA_ROOT = Path(os.environ.get("ROAD_DATA_ROOT", CNN_DATA_ROOT / "Roads"))
RESULTS_ROOT = REPO_ROOT / "results" / "cnn"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"CNN data root:   {CNN_DATA_ROOT}")


# Experiment 3: Sub-Concepts and Distribution Shift

Two sub-experiments. First: do models represent sub-concepts within a class (sleeping vs standing cats)? Second: is KL divergence sensitive to distribution shift within a class (Indian vs Turkish roads)?

## Sub-Experiment 3a: Cat Pose — Sleeping vs Standing

We ask whether the models' pre-final layer activations can separate two semantically distinct sub-categories of the cat class. If pose is encoded as a meaningful internal dimension, sleeping and standing cats should form separable clusters.

### Setup and Feature Extraction

In [ ]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ============================================================
# CELL 2 — Image Loader
# ============================================================
def load_images_from_folder(folder_path, label):
    """Returns list of (PIL image, label, filename)"""
    supported = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    entries = []
    for fname in sorted(os.listdir(folder_path)):
        if fname.lower().endswith(supported):
            img = Image.open(os.path.join(folder_path, fname)).convert('RGB')
            entries.append((img, label, fname))
    return entries

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load both classes
sleeping_data = load_images_from_folder(str(POSE_DATA_ROOT / "Sleeping"), label=0)
standing_data = load_images_from_folder(str(POSE_DATA_ROOT / "Standing"), label=1)
all_data      = sleeping_data + standing_data

labels    = np.array([d[1] for d in all_data])
filenames = [d[2] for d in all_data]

print(f"Sleeping: {len(sleeping_data)} images")
print(f"Standing: {len(standing_data)} images")
print(f"Total   : {len(all_data)} images")


In [ ]:
# ============================================================
# CELL 3 — Feature Extractor (pre-final layer)  [FIXED]
# ============================================================
import torch.nn.functional as F

class MobileNetV2Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.features = base.features   # outputs (B, 1280, 7, 7)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))  # → (B, 1280, 1, 1)
        x = torch.flatten(x, 1)               # → (B, 1280)
        return x

class ResNet50Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Everything except the final FC layer
        self.backbone = nn.Sequential(*list(base.children())[:-1])  # → (B, 2048, 1, 1)

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)   # → (B, 2048)
        return x

mob_backbone = MobileNetV2Extractor().eval().to(device)
res_backbone = ResNet50Extractor().eval().to(device)
print("Models loaded ✓")


In [ ]:
# ============================================================
# CELL 4 — Extract Features & Run PCA
# ============================================================
print("Extracting MobileNetV2 features …")
mob_feats = extract_features(mob_backbone, all_data)

print("Extracting ResNet50 features …")
res_feats = extract_features(res_backbone, all_data)

# PCA → 3D
pca3 = PCA(n_components=3, random_state=42)
mob_pca = pca3.fit_transform(mob_feats)
print(f"MobileNetV2 explained variance (3 PCs): "
      f"{pca3.explained_variance_ratio_.sum()*100:.1f}%")

pca3b = PCA(n_components=3, random_state=42)
res_pca = pca3b.fit_transform(res_feats)
print(f"ResNet50 explained variance (3 PCs): "
      f"{pca3b.explained_variance_ratio_.sum()*100:.1f}%")


### PCA 3D — All Cats (Single Colour)

In [ ]:
# ============================================================
# CELL 5 — Plot Helper  [WHITE BG, DARK MAGMA COLOURS]
# ============================================================

MAGMA_CAT      = "#3b0f70"   # deep magma purple — single colour
MAGMA_SLEEPING = "#8c2981"   # magma mid-purple
MAGMA_STANDING = "#de4968"   # magma warm red-pink

# alternatively try:
# MAGMA_SLEEPING = "#f1605d"
# MAGMA_STANDING = "#feaf77"  for orange feel

def plot_3d_pca(pca_data, labels, title,
                mode="single",
                ax=None, fig=None):
    if ax is None:
        fig = plt.figure(figsize=(8, 6))
        ax  = fig.add_subplot(111, projection='3d')

    if mode == "single":
        ax.scatter(pca_data[:, 0], pca_data[:, 1], pca_data[:, 2],
                   c=MAGMA_CAT, s=60, alpha=0.85,
                   edgecolors='none', label='Cats', depthshade=True)

    elif mode == "split":
        mask_sleep = labels == 0
        mask_stand = labels == 1

        ax.scatter(pca_data[mask_sleep, 0],
                   pca_data[mask_sleep, 1],
                   pca_data[mask_sleep, 2],
                   c=MAGMA_SLEEPING, s=65, alpha=0.85,
                   edgecolors='none', label='Sleeping', depthshade=True)

        ax.scatter(pca_data[mask_stand, 0],
                   pca_data[mask_stand, 1],
                   pca_data[mask_stand, 2],
                   c=MAGMA_STANDING, s=65, alpha=0.85,
                   edgecolors='none', label='Standing', depthshade=True)

    ax.legend(fontsize=9)
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2'); ax.set_zlabel('PC 3')
    ax.set_title(title, fontsize=11, pad=10)
    return fig, ax


In [ ]:
# ============================================================
# CELL 6 — Figure 1: Single Colour  [WHITE BG]
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                         subplot_kw={'projection': '3d'})
fig.suptitle("Pre-Final Layer 3D PCA  ·  All Cats", fontsize=13)

plot_3d_pca(mob_pca, labels, "MobileNetV2", mode="single", ax=axes[0], fig=fig)
plot_3d_pca(res_pca, labels, "ResNet50",    mode="single", ax=axes[1], fig=fig)

plt.tight_layout()
plt.savefig("pca_single_colour.png", dpi=150, bbox_inches='tight')
plt.show()


### PCA 3D — Sleeping vs Standing

In [ ]:
# ============================================================
# CELL 7 — Figure 2: Split by Pose  [WHITE BG]
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                         subplot_kw={'projection': '3d'})
fig.suptitle("Pre-Final Layer 3D PCA  ·  Sleeping vs Standing", fontsize=13)

plot_3d_pca(mob_pca, labels, "MobileNetV2", mode="split", ax=axes[0], fig=fig)
plot_3d_pca(res_pca, labels, "ResNet50",    mode="split", ax=axes[1], fig=fig)

plt.tight_layout()
plt.savefig("pca_split_pose.png", dpi=150, bbox_inches='tight')
plt.show()


### Distance Heatmaps — Sleeping vs Standing

In [ ]:
# ============================================================
# CELL 9 — Pairwise Distance Matrices (Raw + PCA)
# ============================================================
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances

# Raw activations
mob_euc_raw = euclidean_distances(mob_feats)
mob_cos_raw = cosine_distances(mob_feats)
res_euc_raw = euclidean_distances(res_feats)
res_cos_raw = cosine_distances(res_feats)

# 3D PCA projections
mob_euc_pca = euclidean_distances(mob_pca)
mob_cos_pca = cosine_distances(mob_pca)
res_euc_pca = euclidean_distances(res_pca)
res_cos_pca = cosine_distances(res_pca)

print("Distance matrices computed ✓")


In [ ]:
# ============================================================
# CELL 11 — Plot all 8 Heatmaps
# ============================================================
def draw_heatmap(ax, mat, title, cmap="viridis"):
    im = ax.imshow(mat, cmap=cmap, aspect='equal')
    ax.set_xticks([0, 1]); ax.set_xticklabels(class_names, fontsize=10)
    ax.set_yticks([0, 1]); ax.set_yticklabels(class_names, fontsize=10)
    ax.set_title(title, fontsize=10, pad=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    # Annotate each cell with its value
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{mat[i, j]:.4f}",
                    ha='center', va='center',
                    color='white' if mat[i, j] < mat.max() * 0.6 else 'black',
                    fontsize=11, fontweight='bold')

fig, axes = plt.subplots(4, 2, figsize=(12, 22))
fig.suptitle("Mean Pairwise Distances — Class × Class (2×2 Heatmaps)", 
             fontsize=14, y=1.01)

heatmaps = [
    (mob_euc_raw_2x2, "MobileNetV2 · Euclidean · Raw Activations"),
    (res_euc_raw_2x2, "ResNet50    · Euclidean · Raw Activations"),
    (mob_cos_raw_2x2, "MobileNetV2 · Cosine    · Raw Activations"),
    (res_cos_raw_2x2, "ResNet50    · Cosine    · Raw Activations"),
    (mob_euc_pca_2x2, "MobileNetV2 · Euclidean · 3D PCA"),
    (res_euc_pca_2x2, "ResNet50    · Euclidean · 3D PCA"),
    (mob_cos_pca_2x2, "MobileNetV2 · Cosine    · 3D PCA"),
    (res_cos_pca_2x2, "ResNet50    · Cosine    · 3D PCA"),
]

for ax, (mat, title) in zip(axes.flatten(), heatmaps):
    draw_heatmap(ax, mat, title)

plt.tight_layout()
plt.savefig("class_distance_heatmaps.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → class_distance_heatmaps.png")


In [ ]:
# ============================================================
# CELL 11 — Plot all 8 Heatmaps
# ============================================================
def draw_heatmap(ax, mat, title, cmap="viridis"):
    im = ax.imshow(mat, cmap=cmap, aspect='equal')
    ax.set_xticks([0, 1]); ax.set_xticklabels(class_names, fontsize=10)
    ax.set_yticks([0, 1]); ax.set_yticklabels(class_names, fontsize=10)
    ax.set_title(title, fontsize=10, pad=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    # Annotate each cell with its value
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{mat[i, j]:.4f}",
                    ha='center', va='center',
                    color='white' if mat[i, j] < mat.max() * 0.6 else 'black',
                    fontsize=11, fontweight='bold')

fig, axes = plt.subplots(4, 2, figsize=(12, 22))
fig.suptitle("Mean Pairwise Distances — Class × Class (2×2 Heatmaps)", 
             fontsize=14, y=1.01)

heatmaps = [
    (mob_euc_raw_2x2, "MobileNetV2 · Euclidean · Raw Activations"),
    (res_euc_raw_2x2, "ResNet50    · Euclidean · Raw Activations"),
    (mob_cos_raw_2x2, "MobileNetV2 · Cosine    · Raw Activations"),
    (res_cos_raw_2x2, "ResNet50    · Cosine    · Raw Activations"),
    (mob_euc_pca_2x2, "MobileNetV2 · Euclidean · 3D PCA"),
    (res_euc_pca_2x2, "ResNet50    · Euclidean · 3D PCA"),
    (mob_cos_pca_2x2, "MobileNetV2 · Cosine    · 3D PCA"),
    (res_cos_pca_2x2, "ResNet50    · Cosine    · 3D PCA"),
]

for ax, (mat, title) in zip(axes.flatten(), heatmaps):
    draw_heatmap(ax, mat, title)

plt.tight_layout()
plt.savefig("class_distance_heatmaps.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → class_distance_heatmaps.png")


### Mahalanobis Distance — Sleeping vs Standing

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
def load_images_from_folder(folder_path, n=100, img_size=224):
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    images = []
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_exts)][:n]
    
    for fname in files:
        try:
            img = Image.open(os.path.join(folder_path, fname)).convert('RGB')
            images.append(transform(img))
        except Exception as e:
            print(f"Skipping {fname}: {e}")
    
    print(f"Loaded {len(images)} images from {folder_path}")
    return torch.stack(images) if images else None


In [ ]:
def build_resnet_with_hook():
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    model.eval().to(device)
    activations = {}

    def hook_fn(module, input, output):
        activations['pre_final'] = output.detach().cpu().numpy()

    # Hook on avgpool (pre-final layer, before the FC head)
    model.avgpool.register_forward_hook(hook_fn)
    return model, activations

def build_mobilenet_with_hook():
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    model.eval().to(device)
    activations = {}

    def hook_fn(module, input, output):
        activations['pre_final'] = output.detach().cpu().numpy()

    # Hook on the adaptive avg pool (pre-final layer)
    model.features.register_forward_hook(hook_fn)
    return model, activations

resnet_model, resnet_acts = build_resnet_with_hook()
mobilenet_model, mobilenet_acts = build_mobilenet_with_hook()
print("Models ready.")


In [ ]:
def extract_activations(model, activations_dict, image_tensor, batch_size=16):
    all_acts = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(image_tensor), batch_size):
            batch = image_tensor[i:i+batch_size].to(device)
            _ = model(batch)
            act = activations_dict['pre_final']
            # Flatten spatial dims if needed
            act = act.reshape(act.shape[0], -1)
            all_acts.append(act)
    return np.vstack(all_acts)


In [ ]:
DATA_ROOT = str(CNN_DATA_ROOT)  # <-- adjust path if needed
data_folders = {
    "Cats":        os.path.join(DATA_ROOT, "Cats"),
    "Dogs":        os.path.join(DATA_ROOT, "Dogs"),
    "Cars":        os.path.join(DATA_ROOT, "Cars"),
    "Rangoli":     os.path.join(DATA_ROOT, "Rangoli"),
    "Microscopy":  os.path.join(DATA_ROOT, "Microscopy"),
}

data_tensors = {}
for name, path in data_folders.items():
    data_tensors[name] = load_images_from_folder(path, n=200)


In [ ]:
POSE_ROOT = str(POSE_DATA_ROOT)  # <-- adjust path if needed
pose_folders = {
    "Sleeping": os.path.join(POSE_ROOT, "sleeping"),
    "Standing": os.path.join(POSE_ROOT, "standing"),
}

pose_tensors = {}
for name, path in pose_folders.items():
    pose_tensors[name] = load_images_from_folder(path, n=100)


In [ ]:
# Data_100 activations
resnet_data_acts  = {}
mobilenet_data_acts = {}

for name, tensor in data_tensors.items():
    if tensor is not None:
        print(f"Extracting ResNet  activations: {name}")
        resnet_data_acts[name]    = extract_activations(resnet_model,    resnet_acts,    tensor)
        print(f"Extracting MobileNet activations: {name}")
        mobilenet_data_acts[name] = extract_activations(mobilenet_model, mobilenet_acts, tensor)

# Pose_100 activations (ResNet only for bar plot 5 & 6, but we'll do both)
resnet_pose_acts    = {}
mobilenet_pose_acts = {}

for name, tensor in pose_tensors.items():
    if tensor is not None:
        print(f"Extracting ResNet  activations: {name}")
        resnet_pose_acts[name]    = extract_activations(resnet_model,    resnet_acts,    tensor)
        print(f"Extracting MobileNet activations: {name}")
        mobilenet_pose_acts[name] = extract_activations(mobilenet_model, mobilenet_acts, tensor)

print("All activations extracted!")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import mahalanobis

def pointwise_mahal_matrix(acts_dict, keys, n_components=50):
    X_all = np.vstack([acts_dict[k] for k in keys])
    X_all = StandardScaler().fit_transform(X_all)
    n_comp = min(n_components, X_all.shape[0] - 1, X_all.shape[1])
    X_all  = PCA(n_components=n_comp).fit_transform(X_all)

    normed, idx = {}, 0
    for k in keys:
        n = len(acts_dict[k])
        normed[k] = X_all[idx:idx+n]
        idx += n

    n_total = len(X_all)
    pooled_cov = sum(
        np.cov(normed[k].T) * (len(normed[k]) - 1) for k in keys
    ) / (n_total - len(keys))
    pooled_cov += np.eye(pooled_cov.shape[0]) * 1e-6
    VI = np.linalg.inv(pooled_cov)

    n = len(keys)
    D = np.zeros((n, n))
    for i, src in enumerate(keys):
        for j, tgt in enumerate(keys):
            mu_tgt = normed[tgt].mean(axis=0)
            D[i, j] = np.mean([mahalanobis(x, mu_tgt, VI) for x in normed[src]])
    return D


def plot_mahal(ax, D, keys, title):
    n = len(keys)

    im = ax.imshow(D, cmap='viridis', aspect='auto',
                   vmin=D.min(), vmax=D.max(), interpolation='nearest')

    # Annotate every cell
    vrange = D.max() - D.min()
    for i in range(n):
        for j in range(n):
            val      = D[i, j]
            norm_val = (val - D.min()) / (vrange + 1e-9)
            color    = 'white' if norm_val < 0.6 else '#0d0221'
            ax.text(j, i, f"{val:.1f}",
                    ha='center', va='center',
                    fontsize=11, fontweight='bold', color=color)

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(keys, fontsize=11, color='#1e1b4b', fontweight='semibold')
    ax.set_yticklabels(keys, fontsize=11, color='#1e1b4b', fontweight='semibold')
    ax.tick_params(length=0)
    ax.set_xlabel("Target cluster mean", fontsize=9, color='#4b5563', labelpad=6)
    ax.set_ylabel("Source points",       fontsize=9, color='#4b5563', labelpad=6)
    ax.set_title(title, fontsize=12, fontweight='bold', color='#1e1b4b', pad=10)

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8, colors='#374151')
    cbar.set_label("Mahalanobis Distance", fontsize=8, color='#374151')
    cbar.outline.set_linewidth(0)

    # Clean cell borders
    for x in np.arange(-0.5, n, 1):
        ax.axhline(x, color='white', linewidth=2)
        ax.axvline(x, color='white', linewidth=2)

    for spine in ax.spines.values():
        spine.set_visible(False)


# ── Group definitions ──────────────────────────────────────────────────────
cats_dogs_cars = ["Cats", "Dogs", "Cars"]
all_five       = ["Cats", "Dogs", "Cars", "Rangoli", "Microscopy"]
pose_classes   = ["Sleeping", "Standing"]

SAVE_KW = dict(dpi=200, bbox_inches='tight', facecolor='white')

# ── Plot 1 : Cats · Dogs · Cars ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('white')
fig.suptitle("Mahalanobis Distance  ·  Cats · Dogs · Cars",
             fontsize=14, fontweight='bold', color='#1e1b4b', y=1.02)
plot_mahal(ax1, pointwise_mahal_matrix(resnet_data_acts,    cats_dogs_cars), cats_dogs_cars, "ResNet50")
plot_mahal(ax2, pointwise_mahal_matrix(mobilenet_data_acts, cats_dogs_cars), cats_dogs_cars, "MobileNetV2")
fig.tight_layout(w_pad=5)
fig.savefig("mahal1_cats_dogs_cars.png", **SAVE_KW)

# ── Plot 2 : All 5 Classes ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')
fig.suptitle("Mahalanobis Distance  ·  All 5 Classes",
             fontsize=14, fontweight='bold', color='#1e1b4b', y=1.02)
plot_mahal(ax1, pointwise_mahal_matrix(resnet_data_acts,    all_five), all_five, "ResNet50")
plot_mahal(ax2, pointwise_mahal_matrix(mobilenet_data_acts, all_five), all_five, "MobileNetV2")
fig.tight_layout(w_pad=5)
fig.savefig("mahal2_all5.png", **SAVE_KW)

# ── Plot 3 : Sleeping · Standing ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
fig.patch.set_facecolor('white')
fig.suptitle("Mahalanobis Distance  ·  Sleeping · Standing",
             fontsize=14, fontweight='bold', color='#1e1b4b', y=1.02)
plot_mahal(ax1, pointwise_mahal_matrix(resnet_pose_acts,    pose_classes), pose_classes, "ResNet50")
plot_mahal(ax2, pointwise_mahal_matrix(mobilenet_pose_acts, pose_classes), pose_classes, "MobileNetV2")
fig.tight_layout(w_pad=5)
fig.savefig("mahal3_pose.png", **SAVE_KW)

plt.show()
print("Saved → mahal1_cats_dogs_cars.png")
print("        mahal2_all5.png")
print("        mahal3_pose.png")


### KL Divergence — Sleeping vs Standing Baseline

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt

# ── CONFIG ────────────────────────────────────────────────────────────
DATA_DIR  = str(POSE_DATA_ROOT)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_BASE    = 100
N_TEST    = 100

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# ── TRANSFORM ─────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

def load_image(path):
    return transform(Image.open(path).convert("RGB"))

def get_image_paths(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(image_extensions)
    ])

# ── MODELS ────────────────────────────────────────────────────────────
# ResNet-50
print("Loading ResNet-50 ...")
resnet   = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(DEVICE)
resnet.eval()
rn_model = nn.Sequential(*list(resnet.children())[:-1])
rn_model.eval().to(DEVICE)

def rn_get_activation(img_tensor):
    with torch.no_grad():
        feat = rn_model(img_tensor).view(-1)
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
        return feat.cpu().numpy()

# MobileNetV2
print("Loading MobileNetV2 ...")
mobilenet_full = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).to(DEVICE)
mobilenet_full.eval()

class MobileNetFeatures(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)

mn_model = MobileNetFeatures(mobilenet_full)
mn_model.eval().to(DEVICE)

def mn_get_activation(img_tensor):
    with torch.no_grad():
        feat = mn_model(img_tensor).view(-1)
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
        return feat.cpu().numpy()

# ── STEP 1: BUILD SLEEPING BASELINE ───────────────────────────────────
sleeping_paths  = get_image_paths(os.path.join(DATA_DIR, "Sleeping"))
standing_paths  = get_image_paths(os.path.join(DATA_DIR, "Standing"))

def build_baseline(get_act, paths, n=N_BASE):
    feats = []
    for path in paths[:n]:
        img  = load_image(path).unsqueeze(0).to(DEVICE)
        feat = get_act(img)
        feats.append(feat)
    mean_vec = np.stack(feats).mean(axis=0)
    return mean_vec / (mean_vec.sum() + 1e-8)

print("\nBuilding baselines ...")
rn_baseline = build_baseline(rn_get_activation, sleeping_paths)
mn_baseline = build_baseline(mn_get_activation, sleeping_paths)

# ── STEP 2: COMPUTE KL FOR SLEEPING AND STANDING ──────────────────────
def compute_kl(get_act, baseline, paths, n=N_TEST):
    kl_vals = []
    for path in paths[:n]:
        img  = load_image(path).unsqueeze(0).to(DEVICE)
        feat = get_act(img)
        feat = feat / (feat.sum() + 1e-8)
        kl   = entropy(feat + 1e-8, baseline + 1e-8)
        kl_vals.append(kl)
    return kl_vals

print("\nComputing ResNet-50 KL ...")
rn_sleeping_kl = compute_kl(rn_get_activation, rn_baseline,
                             sleeping_paths[N_BASE:N_BASE + N_TEST])
rn_standing_kl = compute_kl(rn_get_activation, rn_baseline,
                             standing_paths[:N_TEST])

print("Computing MobileNetV2 KL ...")
mn_sleeping_kl = compute_kl(mn_get_activation, mn_baseline,
                             sleeping_paths[N_BASE:N_BASE + N_TEST])
mn_standing_kl = compute_kl(mn_get_activation, mn_baseline,
                             standing_paths[:N_TEST])

# ── STEP 3: PRINT RESULTS ─────────────────────────────────────────────
print("\n==============================")
print("FINAL RESULTS")
print("==============================")
print(f"ResNet-50   — Sleeping vs Sleeping baseline: {np.mean(rn_sleeping_kl):.6f}")
print(f"ResNet-50   — Standing vs Sleeping baseline: {np.mean(rn_standing_kl):.6f}")
print(f"MobileNetV2 — Sleeping vs Sleeping baseline: {np.mean(mn_sleeping_kl):.6f}")
print(f"MobileNetV2 — Standing vs Sleeping baseline: {np.mean(mn_standing_kl):.6f}")

# ── STEP 4: DISTRIBUTION SHIFT PLOTS ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

for ax, sleeping_kl, standing_kl, model_name in zip(
    axes,
    [rn_sleeping_kl, mn_sleeping_kl],
    [rn_standing_kl, mn_standing_kl],
    ["ResNet-50", "MobileNetV2"]
):
    ax.hist(sleeping_kl, bins=20, alpha=0.6,
            color='steelblue', label=f"Sleeping (avg={np.mean(sleeping_kl):.4f})")
    ax.hist(standing_kl, bins=20, alpha=0.6,
            color='crimson',   label=f"Standing (avg={np.mean(standing_kl):.4f})")

    ax.axvline(np.mean(sleeping_kl), color='steelblue',
               linestyle='--', linewidth=1.5)
    ax.axvline(np.mean(standing_kl), color='crimson',
               linestyle='--', linewidth=1.5)

    ax.text(np.mean(sleeping_kl), ax.get_ylim()[1] * 0.92,
            f"{np.mean(sleeping_kl):.4f}",
            color='steelblue', ha='center', fontsize=9, fontweight='bold')
    ax.text(np.mean(standing_kl), ax.get_ylim()[1] * 0.82,
            f"{np.mean(standing_kl):.4f}",
            color='crimson', ha='center', fontsize=9, fontweight='bold')

    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.set_xlabel("KL Divergence vs Sleeping Baseline", fontsize=9)
    ax.set_ylabel("Frequency", fontsize=9)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor('#f9f9f9')
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)

plt.suptitle("Distribution Shift — Sleeping vs Standing\n(baseline = first 100 Sleeping images)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("distribution_shift_pose.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── STEP 3: PRINT RESULTS ─────────────────────────────────────────────
print("\n==============================")
print("FINAL RESULTS")
print("==============================")
print(f"ResNet-50   — Sleeping vs Sleeping baseline: {np.mean(rn_sleeping_kl):.6f}")
print(f"ResNet-50   — Standing vs Sleeping baseline: {np.mean(rn_standing_kl):.6f}")
print(f"MobileNetV2 — Sleeping vs Sleeping baseline: {np.mean(mn_sleeping_kl):.6f}")
print(f"MobileNetV2 — Standing vs Sleeping baseline: {np.mean(mn_standing_kl):.6f}")

# ── STEP 4: DISTRIBUTION SHIFT PLOTS ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

for ax, sleeping_kl, standing_kl, model_name in zip(
    axes,
    [rn_sleeping_kl, mn_sleeping_kl],
    [rn_standing_kl, mn_standing_kl],
    ["ResNet-50", "MobileNetV2"]
):
    ax.hist(sleeping_kl, bins=20, alpha=0.6,
            color='#440154', label=f"Sleeping (avg={np.mean(sleeping_kl):.4f})")
    ax.hist(standing_kl, bins=20, alpha=0.6,
            color="#f98e09",   label=f"Standing (avg={np.mean(standing_kl):.4f})")

    ax.axvline(np.mean(sleeping_kl), color='#440154',
               linestyle='--', linewidth=1.5)
    ax.axvline(np.mean(standing_kl), color="#f98e09",
               linestyle='--', linewidth=1.5)

    ax.text(np.mean(sleeping_kl), ax.get_ylim()[1] * 0.92,
            f"{np.mean(sleeping_kl):.4f}",
            color='#440154', ha='center', fontsize=9, fontweight='bold')
    ax.text(np.mean(standing_kl), ax.get_ylim()[1] * 0.82,
            f"{np.mean(standing_kl):.4f}",
            color="#f98e09", ha='center', fontsize=9, fontweight='bold')

    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.set_xlabel("KL Divergence vs Sleeping Baseline", fontsize=9)
    ax.set_ylabel("Frequency", fontsize=9)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor('#f9f9f9')
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)

plt.suptitle("Distribution Shift — Sleeping vs Standing\n(baseline = first 100 Sleeping images)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("distribution_shift_pose.png", dpi=150, bbox_inches='tight')
plt.show()


## Sub-Experiment 3b: Road Images — India vs Turkey

We build a baseline from Indian road images and compute KL divergence for held-out Indian images and Turkish road images against this baseline. A higher divergence for Turkish images indicates that distribution shift is detectable via activation distributions without labels.

### ResNet50 — India Baseline and KL Distributions

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt

############################################################
# CONFIG
############################################################

base_path = str(ROAD_DATA_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_images_avg = 100
num_images_test = 100

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")


############################################################
# LOAD RESNET50
############################################################

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device)
resnet.eval()

# Remove final FC layer → Penultimate activations
model = nn.Sequential(*list(resnet.children())[:-1])


############################################################
# IMAGE TRANSFORMS
############################################################

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


############################################################
# HELPER FUNCTIONS
############################################################

def load_image(path):
    return transform(Image.open(path).convert("RGB"))


def get_activation(img_tensor):
    """
    Extract 2048-d penultimate layer activation
    and min-max normalize to [0,1].
    """
    with torch.no_grad():

        feat = model(img_tensor).view(-1)

        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

        return feat.cpu().numpy()


############################################################
# LOAD INDIA / TURKEY IMAGE PATHS
############################################################

india_folder = os.path.join(base_path, "India")
turkey_folder = os.path.join(base_path, "Turkey")

india_images = sorted([
    os.path.join(india_folder, f)
    for f in os.listdir(india_folder)
    if f.lower().endswith(image_extensions)
])

turkey_images = sorted([
    os.path.join(turkey_folder, f)
    for f in os.listdir(turkey_folder)
    if f.lower().endswith(image_extensions)
])

assert len(india_images) >= 200, "Need at least 200 India images"
assert len(turkey_images) >= 100, "Need at least 100 Turkey images"


############################################################
# STEP 1 — BUILD INDIA BASELINE HISTOGRAM
############################################################

baseline_feats = []

print("Building India baseline histogram...")

for img_path in india_images[:num_images_avg]:

    img = load_image(img_path).unsqueeze(0).to(device)

    feat = get_activation(img)

    baseline_feats.append(feat)

baseline_feats = np.stack(baseline_feats)

# Average activations across 100 India images
india_baseline_hist = baseline_feats.mean(axis=0)

# Normalize to probability distribution
india_baseline_hist = india_baseline_hist / (india_baseline_hist.sum() + 1e-8)


############################################################
# PLOT INDIA BASELINE HISTOGRAM
############################################################

plt.figure(figsize=(12,5))
plt.plot(india_baseline_hist)
plt.title("India Baseline Average Activation Histogram")
plt.xlabel("Feature Dimension")
plt.ylabel("Normalized Activation")
plt.grid()
plt.show()


############################################################
# STEP 2 — INDIA TEST KL
############################################################

india_kl_values = []

print("Computing India test KL divergences...")

for img_path in india_images[num_images_avg:num_images_avg+num_images_test]:

    img = load_image(img_path).unsqueeze(0).to(device)

    feat = get_activation(img)

    feat = feat / (feat.sum() + 1e-8)

    kl = entropy(feat + 1e-8, india_baseline_hist + 1e-8)

    india_kl_values.append(kl)

india_avg_kl = np.mean(india_kl_values)


############################################################
# STEP 3 — TURKEY KL
############################################################

turkey_kl_values = []

print("Computing Turkey KL divergences...")

for img_path in turkey_images[:num_images_test]:

    img = load_image(img_path).unsqueeze(0).to(device)

    feat = get_activation(img)

    feat = feat / (feat.sum() + 1e-8)

    kl = entropy(feat + 1e-8, india_baseline_hist + 1e-8)

    turkey_kl_values.append(kl)

turkey_avg_kl = np.mean(turkey_kl_values)


############################################################
# PRINT RESULTS
############################################################

print("\n==============================")
print("FINAL RESULTS")
print("==============================")

print(f"India Test Avg KL vs India Baseline: {india_avg_kl:.6f}")
print(f"Turkey Avg KL vs India Baseline:     {turkey_avg_kl:.6f}")


############################################################
# PLOT KL DISTRIBUTIONS
############################################################

plt.figure(figsize=(10,5))

plt.hist(india_kl_values,  bins=20, alpha=0.6, label="India Test", color='#3b0f70')
plt.hist(turkey_kl_values, bins=20, alpha=0.6, label="Turkey",     color='darkorange')

plt.axvline(india_avg_kl,  color='#3b0f70', linestyle='--', linewidth=1.5)
plt.axvline(turkey_avg_kl, color='darkorange',   linestyle='--', linewidth=1.5)

plt.text(india_avg_kl,  plt.ylim()[1] * 0.92, f'India avg: {india_avg_kl:.4f}',
         color='#3b0f70', ha='center', fontsize=10, fontweight='bold')
plt.text(turkey_avg_kl, plt.ylim()[1] * 0.82, f'Turkey avg: {turkey_avg_kl:.4f}',
         color='darkorange',   ha='center', fontsize=10, fontweight='bold')

plt.legend()
plt.title("ResNet-50 — KL Divergence Distribution")
plt.xlabel("KL Divergence")
plt.ylabel("Frequency")
plt.grid()
plt.show()


### MobileNetV2 — India Baseline and KL Distributions

In [ ]:
############################################################
# LOAD MOBILENETV2
############################################################
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).to(device)
mobilenet.eval()

# Remove final classifier → penultimate activations (1280-d)
mobilenet_feat_model = nn.Sequential(*list(mobilenet.children())[:-1])

############################################################
# MOBILENET HELPER
############################################################

def get_activation_mobilenet(img_tensor):
    """
    Extract 1280-d penultimate layer activation
    and min-max normalize to [0,1].
    """
    with torch.no_grad():
        feat = mobilenet_feat_model(img_tensor)
        feat = feat.mean([2, 3]).view(-1)          # global avg pool → (1280,)
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
        return feat.cpu().numpy()

############################################################
# STEP 1 — MOBILENET INDIA BASELINE
############################################################

mobilenet_baseline_feats = []

print("Building MobileNetV2 India baseline histogram...")

for img_path in india_images[:num_images_avg]:
    img = load_image(img_path).unsqueeze(0).to(device)
    feat = get_activation_mobilenet(img)
    mobilenet_baseline_feats.append(feat)

mobilenet_baseline_feats = np.stack(mobilenet_baseline_feats)

mobilenet_india_baseline = mobilenet_baseline_feats.mean(axis=0)
mobilenet_india_baseline = mobilenet_india_baseline / (mobilenet_india_baseline.sum() + 1e-8)

############################################################
# STEP 2 — MOBILENET INDIA TEST KL
############################################################

mobilenet_india_kl_values = []

print("Computing MobileNetV2 India test KL divergences...")

for img_path in india_images[num_images_avg:num_images_avg + num_images_test]:
    img = load_image(img_path).unsqueeze(0).to(device)
    feat = get_activation_mobilenet(img)
    feat = feat / (feat.sum() + 1e-8)
    kl = entropy(feat + 1e-8, mobilenet_india_baseline + 1e-8)
    mobilenet_india_kl_values.append(kl)

mobilenet_india_avg_kl = np.mean(mobilenet_india_kl_values)

############################################################
# STEP 3 — MOBILENET TURKEY KL
############################################################

mobilenet_turkey_kl_values = []

print("Computing MobileNetV2 Turkey KL divergences...")

for img_path in turkey_images[:num_images_test]:
    img = load_image(img_path).unsqueeze(0).to(device)
    feat = get_activation_mobilenet(img)
    feat = feat / (feat.sum() + 1e-8)
    kl = entropy(feat + 1e-8, mobilenet_india_baseline + 1e-8)
    mobilenet_turkey_kl_values.append(kl)

mobilenet_turkey_avg_kl = np.mean(mobilenet_turkey_kl_values)

############################################################
# PRINT RESULTS
############################################################

print("\n==============================")
print("MOBILENETV2 FINAL RESULTS")
print("==============================")
print(f"India Test Avg KL vs India Baseline: {mobilenet_india_avg_kl:.6f}")
print(f"Turkey Avg KL vs India Baseline:     {mobilenet_turkey_avg_kl:.6f}")

############################################################
# PLOT KL DISTRIBUTIONS
############################################################

plt.figure(figsize=(10,5))

plt.hist(mobilenet_india_kl_values,  bins=20, alpha=0.6, label="India Test", color='#3b0f70')
plt.hist(mobilenet_turkey_kl_values, bins=20, alpha=0.6, label="Turkey",     color='darkorange')

plt.axvline(mobilenet_india_avg_kl,  color='#3b0f70', linestyle='--', linewidth=1.5)
plt.axvline(mobilenet_turkey_avg_kl, color='darkorange',   linestyle='--', linewidth=1.5)

plt.text(mobilenet_india_avg_kl,  plt.ylim()[1] * 0.92, f'India avg: {mobilenet_india_avg_kl:.4f}',
         color='#3b0f70', ha='center', fontsize=10, fontweight='bold')
plt.text(mobilenet_turkey_avg_kl, plt.ylim()[1] * 0.82, f'Turkey avg: {mobilenet_turkey_avg_kl:.4f}',
         color='darkorange',   ha='center', fontsize=10, fontweight='bold')

plt.legend()
plt.title("MobileNetV2 — KL Divergence Distribution")
plt.xlabel("KL Divergence")
plt.ylabel("Frequency")
plt.grid()
plt.show()
